In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu pypdf gradio groq

In [ ]:
import requests

pdf_url = "https://drive.google.com/file/d/1WdjXAzel69SlTiULSm2QO6v6ufrcktgP/view?usp=drive_link"   # <-- replace this
pdf_path = "document.pdf"

response = requests.get(pdf_url)

with open(pdf_path, "wb") as f:
    f.write(response.content)

print("PDF downloaded!")

In [ ]:
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Total chunks: {len(docs)}")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
from google.colab import userdata
from groq import Groq

# Fetch API key securely
api_key = userdata.get("GROQ_API_KEY")

client = Groq(api_key=api_key)

In [ ]:
def rag_answer(query):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    retrieved_docs = retriever.get_relevant_documents(query)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the context below.
If not found, say "I don't know".

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.choices[0].message.content

    sources = "\n".join([doc.metadata.get("source", "Unknown") for doc in retrieved_docs])

    return f"{answer}\n\nSources:\n{sources}"

In [ ]:
import gradio as gr

def chat(query):
    return rag_answer(query)

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(lines=2, placeholder="Ask something from your PDFs..."),
    outputs="text",
    title="📄 RAG PDF Chatbot (Groq + FAISS + HF)",
)

interface.launch(share=True)